<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E09/mva_2026_E09_01_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_L09_hig_clustering

- 学籍番号 [ ]
- 氏名 [ ]

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from IPython.display import clear_output
import time

from sklearn.mixture import GaussianMixture # GMMのEMアルゴリズム

%pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

##ガウス混合分布データの生成

In [ ]:
# 1. ガウス混合分布から疑似データを生成
np.random.seed(42)
n_samples = 300

# 3つの構成要素のパラメータ設定
means = [[-1, -1], [1, 3], [3, -1]]
covs = [[[2.0, 0.5], [0.5, 2.0]],
        [[1.5, -0.3], [-0.3, 2.0]],
        [[0.8, 0.0], [0.0, 0.8]]]
weights = [0.4, 0.3, 0.3]

X = []
for i in range(n_samples):
    # クラスタの選択
    component = np.random.choice(3, p=weights)
    # 選択されたガウス分布からサンプルを抽出
    sample = np.random.multivariate_normal(means[component], covs[component])
    X.append(sample)
X = np.array(X)

# 生成したデータのプロット
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, edgecolors='k')
plt.title("生成された混合ガウス分布のデータ")
plt.xlabel("x1")
plt.ylabel("x2")
plt.grid(True)
plt.show()

##K-meansによるクラスタリングの可視化

In [ ]:
# 2. K-means アルゴリズムのシミュレーションとアニメーション表示
K = 3

# 初期値を原点 (0, 0) の近くに集中させるための設定
# 各データ点から原点(0,0)への距離を計算
dists_to_origin = np.linalg.norm(X - np.array([0, 0]), axis=1)
# 距離が近い順にインデックスをソートし、近い上位20個の候補からK個をランダムに選択
candidate_idx = np.argsort(dists_to_origin)[:20]
initial_centroids_idx = np.random.choice(candidate_idx, K, replace=False)
centroids = X[initial_centroids_idx]

max_iters = 10
for iteration in range(max_iters):
    # ステップ A: 各データ点に最も近い中心点を割り当て（Eステップ相当）
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)

    # 描画のクリアとプロット
    clear_output(wait=True)
    plt.figure(figsize=(8, 6))

    # クラスタごとに色分けして描画
    colors = ['r', 'g', 'b']
    for k in range(K):
        cluster_data = X[labels == k]
        plt.scatter(cluster_data[:, 0], cluster_data[:, 1], c=colors[k], alpha=0.5, label=f'Cluster {k}')
        # 中心点を描画
        plt.scatter(centroids[k, 0], centroids[k, 1], c=colors[k], marker='X', s=200, edgecolors='black', linewidths=2)

    plt.title(f"K-means イテレーション {iteration + 1} (データの割り当て)")
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.legend()
    plt.grid(True)
    plt.show()
    time.sleep(1.5)

    # ステップ B: 中心点の更新（Mステップ相当）
    new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(K)])

    # 収束判定
    if np.all(centroids == new_centroids):
        print(f"イテレーション {iteration + 1} で収束しました。")
        break
    centroids = new_centroids

## 階層的（凝集型）クラスタリングのステップバイステップ可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform
from IPython.display import clear_output
import time

# japanize_matplotlib がない場合はインストールしてインポートする
try:
    import japanize_matplotlib
except ModuleNotFoundError:
    !pip install japanize_matplotlib
    import japanize_matplotlib

# 視覚的に分かりやすくするため、元のデータからランダムに50点をサンプリングして使用します
np.random.seed(42)
sample_indices = np.random.choice(len(X), size=50, replace=False)
X_sub = X[sample_indices]

# 各データ点に一意のID（初期クラスタID）を割り当て
N = len(X_sub)
clusters = {i: [i] for i in range(N)}

# 距離行列の計算
dist_matrix = squareform(pdist(X_sub))

# 凝集型階層クラスタリングのシミュレーション（単一リンク法：最短距離法によるマージ）
active_clusters = list(clusters.keys())

step = 1
while len(active_clusters) > 3:  # 3つのクラスタにまとまるまでマージを繰り返す
    # 最も近いクラスタのペアを見つける
    min_dist = float('inf')
    to_merge = (None, None)

    for idx1, c1 in enumerate(active_clusters):
        for c2 in active_clusters[idx1+1:]:
            # クラスタ間の最短距離（単一リンク法）を計算
            dists = [dist_matrix[p1, p2] for p1 in clusters[c1] for p2 in clusters[c2]]
            current_dist = min(dists)
            if current_dist < min_dist:
                min_dist = current_dist
                to_merge = (c1, c2)

    c1, c2 = to_merge
    # クラスタをマージ
    clusters[c1].extend(clusters[c2])
    del clusters[c2]
    active_clusters.remove(c2)

    # 描画プロットのアニメーション更新
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(10, 7))

    # クラスタごとに色を割り当ててプロット
    colors = plt.cm.get_cmap('tab10', len(active_clusters))
    for i, c_id in enumerate(active_clusters):
        cluster_points = X_sub[clusters[c_id]]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1], color=colors(i), s=100, label=f'Cluster {i+1}', edgecolors='k')

        # クラスタに複数の点が存在する場合、そのクラスタの重心または代表点の結合をイメージさせるように線を引く、あるいは代表ラベルを置く
        if len(cluster_points) > 1:
            centroid = cluster_points.mean(axis=0)
            ax.text(centroid[0], centroid[1], f'C{i+1}', fontsize=12, fontweight='bold', bbox=dict(facecolor='white', alpha=0.6, boxstyle='round,pad=0.3'))

    ax.set_title(f"階層的クラスタリング ステップ {step} (残りクラスタ数: {len(active_clusters)})")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.grid(True)
    plt.show()

    time.sleep(1.2)
    step += 1

属するデータ点が2つ以上になったクラスターには，Cn のようなラベルが表示されます

最終的なクラスタ構造の関係性を示す「デンドログラム（系統図）」もあわせて描画しておきます。

In [ ]:
# 最終的な樹形図（デンドログラム）のプロット
Z = linkage(X_sub, method='single')

plt.figure(figsize=(12, 6))
dendrogram(Z)
plt.title("デンドログラム (シングルリンク法による階層木)")
plt.xlabel("データ点のインデックス")
plt.ylabel("結合距離")
plt.grid(True, axis='y')
plt.show()